In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
from IPython.display import Image


In [4]:
load_dotenv()

True

In [5]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

In [6]:
class llm_state(TypedDict):
    title:str
    outline:str
    blog:str
    

In [7]:
# create graph for blog writing workflow
graph = StateGraph(llm_state)

In [8]:
def outline_gen(state:llm_state) -> llm_state:
    title = state['title']
    prompt = f"Generate a detailed outline for the title: {title}"
    outline = model.invoke(prompt)
    state['outline'] = outline.content
    return state


In [9]:
def blog_gen(state: llm_state) -> llm_state:
    title = state['title']
    outline = state['outline']
    prompt = f"Write a detailed blog post based on the following title and outline.\nTitle: {title}\nOutline: {outline}"
    blog = model.invoke(prompt)
    state['blog'] = blog.content
    return state


In [10]:
def evaluate_blog(state: llm_state) -> llm_state:
    title = state["title"]
    blog = state["blog"]
    prompt = f"Evaluate the following blog post based on the title. Provide a score from 1 to 10 and explain the reasoning.\n Title:{title}\n Blog: {blog}"
    evaluation = model.invoke(prompt)
    state["evaluation"] = evaluation.content
    return state


In [11]:
# add nodes
graph.add_node("outline_gen", outline_gen)
graph.add_node("blog_gen", blog_gen)
graph.add_node("evaluate_blog", evaluate_blog)

In [12]:
# add edges
graph.add_edge(START, "outline_gen")
graph.add_edge("outline_gen", "blog_gen")   
graph.add_edge("blog_gen", "evaluate_blog")
graph.add_edge("evaluate_blog", END)


In [13]:
# comile the graph
workflow1 = graph.compile()
# execute the graph
initial_state = {'title': "Consequences of Iran and America war"}
final_state = workflow1.invoke(initial_state)
print(final_state)


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 26.770359542s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 24.517962658s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 24
}
]

In [ ]:
print(final_state['outline'])

In [ ]:
print(final_state['blog'])

In [ ]:
# visualize the graph
Image(workflow1.get_graph().draw_mermaid_png())